# **Libraries**

In [ ]:
%run nb_spn_common

# **Functions**

## **Mirror creation functions**

In [ ]:
# --------------------------------------------------------
#  Mirrored Database creation function
# --------------------------------------------------------
def create_mirrored_database(
        access_token: str,
        mirrored_database_name: str,
        source_type: str,
        connection_string: str
) -> None:
    """
    Create a Microsoft Fabric Mirrored Database in the current workspace.

    :param access_token: OAuth2 access token
    :param name: Display name for the Mirrored Database
    :param source_type: Source system type ("SqlDatabase", "SqlManagedInstance", "SqlServer", "AzurePostgreSql", "CosmosDb")
    :param connection_string: Connection string for the mirrored source
    """

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/mirroredDatabases"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": mirrored_database_name,
        "source": {
            "type": source_type,
            "properties": {
                "connectionString": connection_string
            }
        }
    }

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    result = response.json()
    mirror_id = result.get("id")

    print(f"Mirrored Database '{mirrored_database_name}' created. ID: {mirror_id}")
    return mirror_id

# --------------------------------------------------------
#  Mirrored Databricks Catalog creation function
# --------------------------------------------------------
def create_mirrored_databricks_catalog(
        access_token: str,
        databricks_mirror_name: str,
        databricks_url: str,
        databricks_catalog_name: str,
        tenant_id: str,
        client_id: str,
        client_secret: str
):
    """
    Create a Fabric Mirrored Azure Databricks Catalog using a Service Principal.
    """

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/mirroredAzureDatabricksCatalogs"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": databricks_mirror_name,
        "source": {
            "workspaceUrl": databricks_url,
            "catalogName": databricks_catalog_name,
            "authentication": {
                "type": "ServicePrincipal",
                "tenantId": tenant_id,
                "clientId": client_id,
                "clientSecret": client_secret
            }
        }
    }

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Mirrored Databricks Catalog '{databricks_mirror_name}' created successfully using Service Principal auth.")

## **Mirror helper functions**

In [ ]:
# --------------------------------------------------------
#  Valid mirror source types
# --------------------------------------------------------

validate_mirror_source_types = {
    "SqlDatabase", # Azure SQL Database
    "SqlManagedInstance", # Azure SQL Managed Instance
    "SqlServer", # On premise SQL Server
    "AzurePostgreSql", # Azure Postgres SQL
    "CosmosDb" # Azure Cosmos DB NoSQL only
}

def validate_mirror_source_type(source_type: str):
    """
    Validates the source_type for a Fabric Mirrored Database.
    Throws a ValueError if someone decides to freestyle.
    """
    if source_type not in validate_mirror_source_types:
        raise ValueError(
            f"Invalid source_type '{source_type}'. "
            f"Valid options: {', '.join(validate_mirror_source_types)}"
        )


# --------------------------------------------------------
#  Validate database connection function
# --------------------------------------------------------
def validate_sql_connection(connection_string: str):
    """
    Validates that the SQL connection works using the service principal.
    """
    try:
        conn = pyodbc.connect(connection_string, timeout=5)
        cursor = conn.cursor()
        cursor.execute("SELECT TOP 1 name FROM sys.databases")
        row = cursor.fetchone()
        print("Connection validated. Sample result:", row)
        conn.close()
        return True

    except Exception as ex:
        print("Connection validation failed:", ex)
        return False

# --------------------------------------------------------
#  Poll mirror provisioning status
# --------------------------------------------------------
def wait_for_mirror_ready(access_token: str, mirror_id: str, timeout_seconds=300, poll_interval=5):
    """
    Polls Fabric until the mirrored database finishes provisioning.
    Returns the final state.
    """

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/mirroredDatabases/{mirror_id}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        response = requests.get(endpoint, headers=headers)
        response.raise_for_status()
        data = response.json()

        state = data.get("state")
        print(f"Mirror status: {state}")

        # Expected end states
        if state in {"Succeeded", "Ready", "Running"}:
            print("Mirrored database is fully provisioned.")
            return data

        # Possible failure states
        if state in {"Failed", "Error", "Canceled"}:
            raise RuntimeError(f"Mirror provisioning failed with state: {state}")

        time.sleep(poll_interval)

    raise TimeoutError("Timed out waiting for mirrored database to finish provisioning.")

# **Operation**

### **Create mirrored database**

In [ ]:
# --------------------------------------------------------
#  Create Mirrored Database with Service Principal
# --------------------------------------------------------
mirrored_database_name = "MIRRORED_DATABASE"
mirrored_database_type = "SqlDatabase"
mirrored_database_conn = "<DatabaseConnectionString>"

# Validate the source type
validate_mirror_source_type(mirrored_database_type)

# Validate connection before provisioning mirrored database
if not validate_sql_connection(mirrored_database_conn):
    print("Mirror not created due to failed SQL connection test.")
else:
    # Create mirror and get Id
    mirror_id = create_mirrored_database(
        access_token,
        mirrored_database_name,
        mirrored_database_type,
        mirrored_database_conn
    )

    # Wait until provisioned
    final_state = wait_for_mirror_ready(access_token, mirror_id)

    print("Final mirror metadata:")
    print(final_state)

### **Create Databricks catalog mirror**

In [ ]:
# --------------------------------------------------------
#  Create Databricks Mirror with Service Principal
# --------------------------------------------------------

# Fabric Item display name
databricks_mirror_name = "dna_prod"

# Databricks variables
databricks_url = "https://adb-6002889854638527.7.azuredatabricks.net"
databricks_catalog_name = "tuatara"

create_mirrored_databricks_catalog(
    access_token,
    databricks_mirror_name,
    databricks_url,
    databricks_catalog_name,
    tenant_id,
    client_id,
    client_secret
)